In [2]:
# loading dependencies
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import os
from get_person_crop_embedding import get_osnet_1x_embedding
import pandas as pd

# enable GPU access
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# helper function to set vid. capture params
def capture_setter(vid_path):
    cap = cv2.VideoCapture(vid_path)
    frame_width = 1280
    frame_height = 720
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, frame_width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, frame_height)
    return cap

In [4]:
model = YOLO('yolov8n.pt')
model.to(device)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [5]:
# loading terrorist embeddings
terrorist_db = {}

# Load Ajmal Khasab
df_ajmal = pd.read_csv("ajmal_khasab.csv")
ajmal_np = df_ajmal.iloc[0].values.astype("float32")
ajmal_tensor = torch.tensor(ajmal_np)
ajmal_tensor = ajmal_tensor.to(device)
terrorist_db["Ajmal_Khasab"] = ajmal_tensor

# Load Osama Bin Laden
df_osama = pd.read_csv("osama_bin_laden.csv")
osama_np = df_osama.iloc[0].values.astype("float32")
osama_tensor = torch.tensor(osama_np)
osama_tensor = osama_tensor.to(device)
terrorist_db["Osama_Bin_Laden"] = osama_tensor

In [25]:
def vid_to_hot_cache(vid):
    
    cap = capture_setter(vid)
  
    while cap.isOpened():
        ret, frame = cap.read()
        if frame is None or cv2.waitKey(1) & 0xFF == 27:
            break
        else:
            frame_cache = [] # frame level cache
            person_bboxes = model.predict(
                frame,
                imgsz=640,
                conf=0.45,          # confidence threshold
                iou=0.7,            # NMS IoU threshold
                classes=[0],        # only person class
                save=False,         # do not save/visualize internally
                verbose=False
                )
            
            annotated_frame = person_bboxes[0].plot()
            result = person_bboxes[0]
            h, w = frame.shape[:2]

            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes.xyxy.cpu().numpy()
                for box in boxes:
                    x1, y1, x2, y2 = box
                    # clip coords
                    x1 = int(np.clip(x1, 0, w - 1))
                    y1 = int(np.clip(y1, 0, h - 1))
                    x2 = int(np.clip(x2, 0, w - 1))
                    y2 = int(np.clip(y2, 0, h - 1))
                    
                    if x2 <= x1 or y2 <= y1:
                        continue
                    else:
                        crop = frame[y1:y2, x1:x2]
                        returned_osnet_1x_embedding = get_osnet_1x_embedding(crop) # generate embedding
                        frame_cache.append({
                            'embedding': returned_osnet_1x_embedding,
                            'bbox': (x1, y1, x2, y2),
                        })

                        # iterate & similarity match through embeddings in frame cache to highlightv terrorist
                        for person in frame_cache:
                            emb = person['embedding']
                            x1, y1, x2, y2 = person['bbox']
                            
                            best_match = None
                            best_distance = float('inf')
                            threshold = 17.0

                            for name, db_emb in terrorist_db.items():
                                #euclidean distance
                                distance = torch.norm(emb - db_emb, p = 2).item()
                                print(distance)
                                if distance < best_distance:
                                    best_distance = distance
                                    best_match = name
                            if best_distance < threshold:
                                cv2.rectangle(
                                    frame,
                                    (x1, y1),
                                    (x2, y2),
                                    (0, 0, 255),
                                    3
                                )
                                cv2.putText(
                                    frame,
                                    f"{best_match} ({best_distance:.2f})",
                                    (x1, y1 - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX,
                                    0.7,
                                    (0, 0, 255),
                                    2
                                )
                                
                cv2.imshow('Video Inference', frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
    cap.release()
    cv2.destroyAllWindows()

In [26]:
vid = 'videos/color_augmented.mp4'
vid_to_hot_cache(vid)

13.000028610229492
14.22168254852295
13.000028610229492
14.22168254852295
17.14788246154785
17.804990768432617
13.000028610229492
14.22168254852295
17.14788246154785
17.804990768432617
17.774850845336914
17.61985206604004
13.094941139221191
14.289631843566895
13.094941139221191
14.289631843566895
17.128700256347656
17.926076889038086
13.094941139221191
14.289631843566895
17.128700256347656
17.926076889038086
17.576879501342773
17.39752960205078
13.094941139221191
14.289631843566895
13.094941139221191
14.289631843566895
17.150175094604492
17.940881729125977
13.094941139221191
14.289631843566895
17.150175094604492
17.940881729125977
17.576879501342773
17.39752960205078
13.102348327636719
14.285691261291504
13.102348327636719
14.285691261291504
17.19532585144043
17.93649673461914
13.102348327636719
14.285691261291504
17.19532585144043
17.93649673461914
17.576879501342773
17.39752960205078
13.102348327636719
14.285691261291504
13.102348327636719
14.285691261291504
17.148239135742188
17.938